In [1]:
import pandas as pd

Note that the fastq files for SP-CEA-F15 are missing in the archive. Either the data transfer failed or something else went wrong.

In [2]:
# Read file that was provided by Sabrina
df = pd.read_csv("../../../Data/NameOfDataset_1/Meta/2019-04-08_CEA_CEF_Manuscript_pt.table_Encore.csv")
df

,Sample ID,Patient,TimePoint,Treatment,RUN
0,SP-CEA-D01,P452,Day 0,-,RUN10
1,SP-CEA-D02,P452,Day 0,-,RUN10
2,SP-CEA-D03,P452,Day 0,-,RUN10
3,SP-CEA-D04,P452,Day 0,-,RUN10
4,SP-CEA-D05,P452,Day 3,CEF priming,RUN10
...,...,...,...,...,...
76,SP-CEA-F35,P452,Day10,IL4,RUN16
77,SP-CEA-F36,P452,Day10,IL4,RUN16
78,SP-CEA-F37,P452,Day10,IL2+IL4,RUN16
79,SP-CEA-F38,P452,Day10,IL2+IL4,RUN16


In [3]:
# Read list with fastq files
myfiles = pd.read_csv("./SAMPLES-CEA-TCRb", sep="/", header=None)
myfiles = myfiles.rename(columns={4:'origRun', 6:'filename'})
cols = ['origRun', 'filename']
myfiles = myfiles[cols]
myfiles.head()

,origRun,filename
0,run10-20161031-miseq,SP-CEA-B17_S165_L001_R1_001.fastq.gz
1,run10-20161031-miseq,SP-CEA-B17_S165_L001_R2_001.fastq.gz
2,run10-20161031-miseq,SP-CEA-B19_S166_L001_R1_001.fastq.gz
3,run10-20161031-miseq,SP-CEA-B19_S166_L001_R2_001.fastq.gz
4,run10-20161031-miseq,SP-CEA-B22_S167_L001_R1_001.fastq.gz


In [4]:
getRun = lambda x: x.split("-")[0].upper()
print("getRun:", getRun('run10-test-blah'))
getSampleNr = lambda x: x.split("_")[1]
print("getSampleNr:", getSampleNr('SP-CEA-B17_S165_L001_R1_001.fastq.gz'))
getSample = lambda x: x.split("_")[0]
def matchSample (x):
    if x[:2] == "SP":
        return(x)
    else:
        return("SP-" + x)
print("getSample:", getSample('SP-CEA-B17_S165_L001_R1_001.fastq.gz'))
print("matchSample:", matchSample('blep_S165_L001_R1_001.fastq.gz'))

getRun: RUN10
getSampleNr: S165
getSample: SP-CEA-B17
matchSample: SP-blep_S165_L001_R1_001.fastq.gz


In [5]:
# Extract run, sample and sample number from file names
myfiles["Run"] = [x for x in map(getRun, myfiles['origRun'])]
myfiles["Sample name"] = [x for x in map(getSample, myfiles['filename'])]
myfiles["Sample match"] = [x for x in map(matchSample, myfiles['Sample name'])]
myfiles["Samples order"] = [x for x in map(getSampleNr, myfiles['filename'])]
myfiles.head()

,origRun,filename,Run,Sample name,Sample match,Samples order
0,run10-20161031-miseq,SP-CEA-B17_S165_L001_R1_001.fastq.gz,RUN10,SP-CEA-B17,SP-CEA-B17,S165
1,run10-20161031-miseq,SP-CEA-B17_S165_L001_R2_001.fastq.gz,RUN10,SP-CEA-B17,SP-CEA-B17,S165
2,run10-20161031-miseq,SP-CEA-B19_S166_L001_R1_001.fastq.gz,RUN10,SP-CEA-B19,SP-CEA-B19,S166
3,run10-20161031-miseq,SP-CEA-B19_S166_L001_R2_001.fastq.gz,RUN10,SP-CEA-B19,SP-CEA-B19,S166
4,run10-20161031-miseq,SP-CEA-B22_S167_L001_R1_001.fastq.gz,RUN10,SP-CEA-B22,SP-CEA-B22,S167


In [6]:
# Keep only the necessary columns and remove duplicate rows
cols = ['Run', 'Sample match', 'Sample name', 'Samples order']
myfiles = myfiles[cols].drop_duplicates()
myfiles.head()

,Run,Sample match,Sample name,Samples order
0,RUN10,SP-CEA-B17,SP-CEA-B17,S165
2,RUN10,SP-CEA-B19,SP-CEA-B19,S166
4,RUN10,SP-CEA-B22,SP-CEA-B22,S167
6,RUN10,SP-CEA-C17,SP-CEA-C17,S168
8,RUN10,SP-CEA-C19,SP-CEA-C19,S41


In [7]:
# Merge tables
df_new = pd.merge(df, myfiles, how='inner', left_on='Sample ID', right_on='Sample match')

In [8]:
df_new["Project"] = "CEA-CEF"
df_new["Species"] = "human"
df_new["Chain"] = "TCRb"
df_new["UMI (yes or no)"] = "yes"
df_new["HEK cells (yes or no)"] = "no"
df_new["RACE (yes or no)"] = "no"
df_new["Constant (yes or no)"] = "no"
df_new

,Sample ID,Patient,TimePoint,Treatment,RUN,Run,Sample match,Sample name,Samples order,Project,Species,Chain,UMI (yes or no),HEK cells (yes or no),RACE (yes or no),Constant (yes or no)
0,SP-CEA-D01,P452,Day 0,-,RUN10,RUN10,SP-CEA-D01,SP-CEA-D01,S66,CEA-CEF,human,TCRb,yes,no,no,no
1,SP-CEA-D02,P452,Day 0,-,RUN10,RUN10,SP-CEA-D02,SP-CEA-D02,S67,CEA-CEF,human,TCRb,yes,no,no,no
2,SP-CEA-D03,P452,Day 0,-,RUN10,RUN10,SP-CEA-D03,SP-CEA-D03,S68,CEA-CEF,human,TCRb,yes,no,no,no
3,SP-CEA-D04,P452,Day 0,-,RUN10,RUN10,SP-CEA-D04,SP-CEA-D04,S69,CEA-CEF,human,TCRb,yes,no,no,no
4,SP-CEA-D05,P452,Day 3,CEF priming,RUN10,RUN10,SP-CEA-D05,SP-CEA-D05,S45,CEA-CEF,human,TCRb,yes,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SP-CEA-F35,P452,Day10,IL4,RUN16,RUN16,SP-CEA-F35,CEA-F35,S177,CEA-CEF,human,TCRb,yes,no,no,no
76,SP-CEA-F36,P452,Day10,IL4,RUN16,RUN16,SP-CEA-F36,CEA-F36,S132,CEA-CEF,human,TCRb,yes,no,no,no
77,SP-CEA-F37,P452,Day10,IL2+IL4,RUN16,RUN16,SP-CEA-F37,CEA-F37,S127,CEA-CEF,human,TCRb,yes,no,no,no
78,SP-CEA-F38,P452,Day10,IL2+IL4,RUN16,RUN16,SP-CEA-F38,CEA-F38,S200,CEA-CEF,human,TCRb,yes,no,no,no


In [9]:
df_new.to_csv("../../../Data/NameOfDataset_1/Meta/20261102-PT-table.csv", index=None)